In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ARC LAB — Data Quality Checks
# MAGIC **Validates Bronze, Silver, and Gold tables after each pipeline run.**
# MAGIC
# MAGIC Checks:
# MAGIC - Row counts are within expected range
# MAGIC - No nulls in critical columns
# MAGIC - Value ranges are realistic
# MAGIC - Data freshness (no stale data)
# MAGIC - Silver has no duplicate 5-min buckets
# MAGIC - Gold has expected number of hour slots

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Config

# COMMAND ----------

from pyspark.sql import functions as F
from datetime import datetime, timezone, timedelta

CATALOG       = "bootcamp_students"
BRONZE_TABLE  = f"{CATALOG}.bronze.watttime_raw"
SILVER_TABLE  = f"{CATALOG}.silver.watttime_clean"
GOLD_TABLE    = f"{CATALOG}.gold.carbon_intensity_hourly"

# Expected values
EXPECTED_SILVER_MIN    = 1800    # minimum acceptable Silver row count
EXPECTED_GOLD_ROWS     = 48      # 24 hours x 2 (weekday + weekend)
MAX_INTENSITY          = 2000    # lbs CO2/MWh upper bound
MIN_INTENSITY          = 0       # lbs CO2/MWh lower bound
FRESHNESS_HOURS        = 48      # Silver data must be no older than this

passed = []
failed = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"  ✓ PASS — {name} {detail}")
    else:
        failed.append(name)
        print(f"  ✗ FAIL — {name} {detail}")

print("DQ checks initialized.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Bronze Checks

# COMMAND ----------

print("=== BRONZE ===\n")

bronze = spark.table(BRONZE_TABLE)
bronze_count = bronze.count()

check("Bronze row count > 0",
      bronze_count > 0,
      f"({bronze_count:,} rows)")

check("Bronze has lbs_co2_per_mwh records",
      bronze.filter(F.col("units") == "lbs_co2_per_mwh").count() > 0)

check("Bronze region_code not null",
      bronze.filter(F.col("region_code").isNull()).count() == 0)

check("Bronze ts_utc not null",
      bronze.filter(F.col("ts_utc").isNull()).count() == 0)

check("Bronze value not null",
      bronze.filter(F.col("value").isNull()).count() == 0)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Silver Checks

# COMMAND ----------

print("\n=== SILVER ===\n")

silver = spark.table(SILVER_TABLE)
silver_count = silver.count()

check("Silver row count above minimum",
      silver_count >= EXPECTED_SILVER_MIN,
      f"({silver_count:,} rows, minimum {EXPECTED_SILVER_MIN:,})")

check("Silver has no WARN or FAIL records",
      silver.filter(F.col("quality_flag") != "PASS").count() == 0)

check("Silver lbs_co2_per_mwh not null",
      silver.filter(F.col("lbs_co2_per_mwh").isNull()).count() == 0)

check("Silver intensity within valid range",
      silver.filter(
          (F.col("lbs_co2_per_mwh") < MIN_INTENSITY) |
          (F.col("lbs_co2_per_mwh") > MAX_INTENSITY)
      ).count() == 0,
      f"(all values between {MIN_INTENSITY} and {MAX_INTENSITY})")

check("Silver ts_5min_bucket not null",
      silver.filter(F.col("ts_5min_bucket").isNull()).count() == 0)

# Duplicate check — should be zero dupes on region + signal + bucket
dupe_count = silver.groupBy("region_code", "signal_type", "ts_5min_bucket") \
    .count().filter(F.col("count") > 1).count()
check("Silver has no duplicate 5-min buckets",
      dupe_count == 0,
      f"({dupe_count} duplicates found)")

# Freshness check
latest_ts = silver.agg(F.max("ts_utc")).collect()[0][0]
now_utc   = datetime.now(timezone.utc).replace(tzinfo=None)
if latest_ts:
    age_hours = (now_utc - latest_ts).total_seconds() / 3600
    check("Silver data is fresh",
          age_hours <= FRESHNESS_HOURS,
          f"(latest record is {age_hours:.1f} hours old, limit {FRESHNESS_HOURS}h)")
else:
    check("Silver data is fresh", False, "(no records found)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Gold Checks

# COMMAND ----------

print("\n=== GOLD ===\n")

gold = spark.table(GOLD_TABLE)
gold_count = gold.count()

check("Gold row count matches expected",
      gold_count == EXPECTED_GOLD_ROWS,
      f"({gold_count} rows, expected {EXPECTED_GOLD_ROWS})")

check("Gold scheduling_opportunity_score not null",
      gold.filter(F.col("scheduling_opportunity_score").isNull()).count() == 0)

check("Gold has low-carbon windows flagged",
      gold.filter(F.col("is_low_carbon_window") == True).count() > 0)

check("Gold has high-carbon windows flagged",
      gold.filter(F.col("is_high_carbon_window") == True).count() > 0)

check("Gold avoidable_lbs_co2_per_mwh > 0",
      gold.agg(F.min("avoidable_lbs_co2_per_mwh")).collect()[0][0] > 0)

check("Gold covers all 24 hours",
      gold.select("hour_of_day").distinct().count() == 24)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Summary

# COMMAND ----------

print("\n" + "="*50)
print(f"  DQ SUMMARY")
print("="*50)
print(f"  PASSED : {len(passed)}")
print(f"  FAILED : {len(failed)}")
print("="*50)

if failed:
    print("\n  ✗ FAILED CHECKS:")
    for f in failed:
        print(f"    - {f}")
    raise Exception(f"DQ checks failed: {len(failed)} issue(s) found. See above.")
else:
    print("\n  ✅ All checks passed. Pipeline is healthy.")

DQ checks initialized.
=== BRONZE ===

  ✓ PASS — Bronze row count > 0 (2,018 rows)
  ✓ PASS — Bronze has lbs_co2_per_mwh records 
  ✓ PASS — Bronze region_code not null 
  ✓ PASS — Bronze ts_utc not null 
  ✓ PASS — Bronze value not null 

=== SILVER ===

  ✓ PASS — Silver row count above minimum (2,016 rows, minimum 1,800)
  ✓ PASS — Silver has no WARN or FAIL records 
  ✓ PASS — Silver lbs_co2_per_mwh not null 
  ✓ PASS — Silver intensity within valid range (all values between 0 and 2000)
  ✓ PASS — Silver ts_5min_bucket not null 
  ✓ PASS — Silver has no duplicate 5-min buckets (0 duplicates found)
  ✓ PASS — Silver data is fresh (latest record is 30.3 hours old, limit 48h)

=== GOLD ===

  ✓ PASS — Gold row count matches expected (48 rows, expected 48)
  ✓ PASS — Gold scheduling_opportunity_score not null 
  ✓ PASS — Gold has low-carbon windows flagged 
  ✓ PASS — Gold has high-carbon windows flagged 
  ✓ PASS — Gold avoidable_lbs_co2_per_mwh > 0 
  ✓ PASS — Gold covers all 24 hou